In [ ]:
%pip install confluent-kafka fastavro faker google-auth urllib3

In [ ]:
import base64
import datetime
import json
import time
import urllib3
import google.auth

from google.auth.transport.urllib3 import Request


def encode(source):
    return (
        base64.urlsafe_b64encode(source.encode("utf-8"))
        .decode("utf-8")
        .rstrip("=")
    )


class TokenProvider:

    HEADER = json.dumps({
        "typ": "JWT",
        "alg": "GOOG_OAUTH2_TOKEN"
    })

    def __init__(self):
        self.credentials, self.project = google.auth.default()
        self.http_client = urllib3.PoolManager()

        print("ADC project:", self.project)
        print("Credentials loaded:", type(self.credentials).__name__)

    def get_credentials(self):
        if not self.credentials.valid:
            self.credentials.refresh(Request(self.http_client))

        return self.credentials

    def get_jwt(self, creds):
        token_data = {
            "exp": creds.expiry.replace(
                tzinfo=datetime.timezone.utc
            ).timestamp(),

            "iat": datetime.datetime.now(
                datetime.timezone.utc
            ).timestamp(),

            "iss": "Google",
            "sub": creds.service_account_email
        }

        return json.dumps(token_data)

    def get_token(self, args):
        creds = self.get_credentials()

        token = ".".join([
            encode(self.HEADER),
            encode(self.get_jwt(creds)),
            encode(creds.token)
        ])

        expiry_utc = creds.expiry.replace(
            tzinfo=datetime.timezone.utc
        )

        now_utc = datetime.datetime.now(
            datetime.timezone.utc
        )

        expiry_seconds = (
            expiry_utc - now_utc
        ).total_seconds()

        return token, time.time() + expiry_seconds

In [ ]:
import fastavro
import io

avro_schema = {
    "type": "record",
    "name": "CardTransactionEvent",
    "namespace": "com.fintech.lakehouse.streaming",

    "fields": [
        {"name": "transaction_id", "type": "string"},
        {"name": "customer_id", "type": "string"},
        {"name": "card_type", "type": "string"},
        {"name": "timestamp", "type": "long"},
        {"name": "amount", "type": "double"},
        {"name": "currency", "type": "string", "default": "USD"},
        {"name": "merchant_name", "type": "string"},
        {"name": "merchant_category", "type": "string"},
        {"name": "location_country", "type": "string"},
        {"name": "is_international", "type": "boolean"},
        {"name": "device_fingerprint", "type": "string"},
        {"name": "fraud_risk_score", "type": "float"}
    ]
}

parsed_schema = fastavro.parse_schema(avro_schema)

In [ ]:
BOOTSTRAP_SERVERS = (
    "bootstrap-m2fh0vmmtugrztbwci63yeoe6."
    "8d4c02221.us-central1.managedkafka.s.cloud.goog:9092"
)

TOPIC_NAME = "card-transactions-topic"

In [ ]:
from confluent_kafka import Producer

token_provider = TokenProvider()

producer_config = {
    "bootstrap.servers": BOOTSTRAP_SERVERS,

    "security.protocol": "SASL_SSL",
    "sasl.mechanisms": "OAUTHBEARER",

    "oauth_cb": token_provider.get_token,

    "acks": "all",
    "enable.idempotence": True
}

producer = Producer(producer_config)

print("Kafka producer initialized.")

ADC project: fintech-lakehouse-dev
Credentials loaded: Credentials
Kafka producer initialized.


In [ ]:
import time
import random
import uuid

from datetime import datetime
from faker import Faker

fake = Faker()

def generate_transaction():

    is_international = random.random() < 0.2

    record = {
        "transaction_id": str(uuid.uuid4()),

        "customer_id": f"CUST-{random.randint(1000, 9999)}",

        "card_type": random.choice([
            "Visa", "MasterCard", "RuPay", "Amex"
        ]),

        "timestamp": int(time.time() * 1000),

        "amount": round(random.uniform(10.0, 1500.0), 2),

        "currency": random.choice(["INR", "INR", "INR", "USD"]),

        "merchant_name": fake.company(),

        "merchant_category": random.choice([
            "retail",
            "electronics",
            "travel",
            "dining",
            "grocery"
        ]),

        "location_country": (
            fake.country_code() if is_international else "IN"
        ),

        "is_international": is_international,

        "device_fingerprint": str(uuid.uuid4())[:8],

        "fraud_risk_score": round(random.random(), 2)
    }

    return record

In [ ]:
def delivery_report(err, msg):

    if err is not None:
        print(f"Delivery failed: {err}")

    else:
        print(
            f"Delivered | "
            f"Topic: {msg.topic()} | "
            f"Partition: {msg.partition()} | "
            f"Offset: {msg.offset()}"
        )

In [ ]:
print(f"Starting producer: {TOPIC_NAME}")

try:

    while True:

        record = generate_transaction()

        # Serialize Python dictionary into Avro binary
        bytes_writer = io.BytesIO()

        fastavro.schemaless_writer(
            bytes_writer,
            parsed_schema,
            record
        )

        avro_bytes = bytes_writer.getvalue()

        # Publish to Kafka
        producer.produce(
            topic=TOPIC_NAME,
            key=record["customer_id"].encode("utf-8"),
            value=avro_bytes,
            callback=delivery_report
        )

        # Serve delivery callbacks
        producer.poll(0)

        print(
            f"Produced: {record['transaction_id']} | "
            f"Customer: {record['customer_id']} | "
            f"Amount: {record['amount']}"
        )

        time.sleep(1)

except KeyboardInterrupt:

    print("Stopping producer...")

finally:

    remaining = producer.flush(timeout=30)

    print(f"Undelivered messages remaining: {remaining}")
    print("Producer stopped.")

Starting producer: card-transactions-topic
Produced: d2f89cfa-0fd0-48fb-b5d4-4891f55c1748 | Customer: CUST-9584 | Amount: 1311.38
Delivered | Topic: card-transactions-topic | Partition: 1 | Offset: 953
Produced: 8e0929de-dffd-47ba-9e49-fc12b4412961 | Customer: CUST-4148 | Amount: 534.41
Delivered | Topic: card-transactions-topic | Partition: 2 | Offset: 971
Produced: f00479bb-6869-4ecc-88ba-32307c5ca8a8 | Customer: CUST-4475 | Amount: 1204.01
Delivered | Topic: card-transactions-topic | Partition: 2 | Offset: 972
Produced: 19917a87-841f-4dd3-9958-bc751d0581b4 | Customer: CUST-7149 | Amount: 138.79
Delivered | Topic: card-transactions-topic | Partition: 0 | Offset: 941
Produced: 6b09508a-a186-4893-9f12-bea100d1888c | Customer: CUST-8263 | Amount: 1170.3
Delivered | Topic: card-transactions-topic | Partition: 0 | Offset: 942
Produced: 5523f910-ee1d-4f94-bbf4-3d1d0bb842d4 | Customer: CUST-4640 | Amount: 1443.31
Delivered | Topic: card-transactions-topic | Partition: 2 | Offset: 973
Produc